# dim_date

Kalenderdimensie gegenereerd uit `[MIN(order_ts), MAX(order_ts)]` over de huidige rijen van `INTEGRATION.DWH_ORDER_HEADER` (`WA_ISCURR = 1`) via `SEQUENCE` + `EXPLODE`, zodat ook dagen zonder orders aanwezig zijn (geen gaten in tijdseries). De surrogate `MK_DATE` is de datum als `yyyymmdd` integer (ADR-0018).

Georkestreerd door `views/07_apply_views.ipynb`.

In [ ]:
%sql
CREATE WIDGET TEXT catalog DEFAULT "DEMO";
USE CATALOG ${catalog};

CREATE OR REPLACE VIEW ${catalog}.DATAMART.DIM_DATE AS
WITH bounds AS (
  SELECT
    MIN(CAST(order_ts AS DATE)) AS min_date,
    MAX(CAST(order_ts AS DATE)) AS max_date
  FROM ${catalog}.INTEGRATION.DWH_ORDER_HEADER
  WHERE WA_ISCURR = 1
),
calendar AS (
  SELECT EXPLODE(SEQUENCE(min_date, max_date, INTERVAL 1 DAY)) AS full_date
  FROM bounds
)
SELECT
  CAST(date_format(full_date, 'yyyyMMdd') AS INT) AS MK_DATE,
  full_date,
  year(full_date) AS year,
  quarter(full_date) AS quarter,
  month(full_date) AS month,
  day(full_date) AS day,
  date_format(full_date, 'MMMM') AS month_name,
  date_format(full_date, 'EEEE') AS day_name,
  dayofweek(full_date) AS day_of_week,
  weekofyear(full_date) AS week_of_year,
  CASE WHEN dayofweek(full_date) IN (1, 7) THEN 1 ELSE 0 END AS is_weekend,
  CAST(date_trunc('month', full_date) AS DATE) AS year_month_start,
  CAST(date_trunc('quarter', full_date) AS DATE) AS year_quarter_start,
  CAST(date_trunc('year', full_date) AS DATE) AS year_start
FROM calendar